# Model

## Characteristics

- autocorrelated data
- time series data

## Tasks

- feature selection (w/ which technic?) (avoid data leakage of data that is not available when planning)
- choose model (only go for one approach, show why others were not pursued)
- avoid overfitting (how?)

## Objectives

- predict on a daily basis the amount of standby drivers efficiently (maximize rate of called in standby drivers)
- minimize days w/ too little drivers (&rarr; dafted drivers needed) (might result in very little days w/ way to many missing drivers, look out for that)
- -> exploitation rate of n_sby for days w/ sby_need/n_sby < 1 and amount of days w/ sby_need/n_sby > 1 

## Restrictions

- plan will be created on the 15th for the following month -> last half of month should not be included in training data

## Further Requirements

- discuss feature importance to increase trust in model (not applicable)
- make predictions as interpretable as possible
- detailed failure analysis to asses situations for which model is not suited
    - plot error in histogram (should be gauss if accumulation somewhere inspect those samples and look for commonalities)
- look up script for only using 90th percentile (for more robustness)

In [ ]:
# overall structure (_pred for predicted features)
# n_sby_pred = n_work_pred + n_sick_pred - n_duty (transformed from calculation of n_work above)
#   n_work_pred predicted by linear regression of calls_pred
#       calls_pred is predicted by time series prediction
#   n_sick_pred is predicted by time series prediction
# o use quantiles (95%) of prophet models

# TODO:
# o rename this notebook to sub_models and create new notebook w/ model.ipynb
# o develop model
#   - SARIMA:
#       Pros: interpretable, provides confidence intervals, clear trend and seasonality
#       Cons: maybe less powerful, no external features, maybe seasonality to complex, only for smaller datasets, problem: yearly seasonality but daily output does not work
#   - XGBoost/NNs: Problem: need many lags (1, 2, 3, 7, 30, 365, ...) because FFT shows influence
#   - Prophet: additive model, high interpretability due to its decomposable components

# o make outputs nicer
#   o no verbose output, error warnings
#   o improved plots w/ reasonable size, grid, title, labels, uncluttered x-axis-labels, ...

In [ ]:
# imports
import os
import numpy as np
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt
import logging
import warnings

%matplotlib ipympl

# import datasets
path = os.path.dirname(os.getcwd())
df_dataset = pd.read_pickle(os.path.join(path, "resources", "df_dataset.pkl"))
df_calls = df_dataset[["date", "calls"]]
df_perc_sick = df_dataset[["date", "perc_sick"]]

# rename columns for prophet
df_calls = df_calls.rename(columns={"date": "ds", "calls": "y"})
df_perc_sick = df_perc_sick.rename(columns={"date": "ds", "perc_sick": "y"})

# split datasets in train and test set
split_date = "2018-05-15" # so test dataset includes every seasonality
df_calls_train = df_calls[df_calls["ds"] < split_date]
df_calls_test = df_calls[df_calls["ds"] >= split_date]
df_perc_sick_train = df_perc_sick[df_perc_sick["ds"] < split_date]
df_perc_sick_test = df_perc_sick[df_perc_sick["ds"] >= split_date]

# avoid verbose output of prophet and cmdstanpy
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
# ignore warnings of prophet
warnings.filterwarnings(
    "ignore",
    message=".*Series.view is deprecated.*",
    category=FutureWarning,
    module="prophet.plot"
)

## Train model to predict calls

In [ ]:
# shows that change points represent trend well (no need for optimization of this hyperparameter)
from prophet.plot import add_changepoints_to_plot

m = Prophet(interval_width=0.95)
m.fit(df_calls)

forecast = m.predict(df_calls)
fig, ax = plt.subplots(figsize=(6, 4))
m.plot(forecast, ax=ax)
add_changepoints_to_plot(ax, m, forecast)
ax.set_title("Changepoints of Prophet model for calls")
ax.plot([], [], "r--", label="Changepoints")
ax.plot([], [], "r-", label="Trend")
ax.legend()
fig.tight_layout()

In [ ]:
# first model w/o hyperparameter optimization
m = Prophet(interval_width=0.95)
m.fit(df_calls_train)

forecast = m.predict(df_calls_test)
print("MAPE: ", np.mean(np.abs(forecast["yhat"].values - df_calls_test["y"].values) / df_calls_test["y"].values))
fig = m.plot_components(forecast)
fig.suptitle("Components of Prophet model for calls")
fig.tight_layout()
fig, ax = plt.subplots(figsize=(6, 3))
m.plot(forecast, ax=ax)
ax.scatter(df_calls_test["ds"], df_calls_test["y"], color="black", marker=".")
ax.set_title("Forecast vs Actuals")
ax.legend()
fig.tight_layout()

In [ ]:
# tryout of adding monthly seasonality
#   determined in FFT, but only results in 1% improvement in MAPE and therefore not worth the extra complexity
m = Prophet(interval_width=0.95)
m.add_seasonality(name="monthly", period=30.5, fourier_order=10)
m.fit(df_calls)

forecast = m.predict(df_calls_test)
print("MAPE: ", np.mean(np.abs(forecast["yhat"].values - df_calls_test["y"].values) / df_calls_test["y"].values))
fig = m.plot_components(forecast)
fig.suptitle("Components of Prophet model for calls")
fig.tight_layout()

In [ ]:
# cross validation for optimization of hyperparameters (weekly and yearly seasonality prior)
#   horizon: duration of use-case (always starting from the 15th, depending on starting month: 16d+30d or 15d+31d)
#   initial: 1 year to ensure that model can learn every seasonality
#   period: repeat for every month
import itertools
from prophet.diagnostics import cross_validation, performance_metrics

# generate scale_prior for weekly and yearly seasonality
scale_priors = [0.01, 0.1, 1, 10]
scale_prior_combs = list(itertools.product(scale_priors, repeat=2))

# use cross validation to evaluate all combinations of scale_prior
tuning_results = pd.DataFrame(columns=["weekly_seasonality_prior", "yearly_seasonality_prior", "mape"])
for scale_prior_comb in scale_prior_combs:
    m = Prophet(
        interval_width=0.95, yearly_seasonality=scale_prior_comb[0], weekly_seasonality=scale_prior_comb[1]
    ).fit(df_calls_train)
    df_cv = cross_validation(m, initial="365.25 days", period="30.5 days", horizon="45 days", disable_tqdm=True)
    df_p = performance_metrics(df_cv)

    # store results
    tuning_results.loc[len(tuning_results)] = {
        "yearly_seasonality_prior": scale_prior_comb[0],
        "weekly_seasonality_prior": scale_prior_comb[1],
        "mape": np.mean(df_p["mape"].values),
    }

# show best results
display(tuning_results.sort_values("mape").head())

In [ ]:
# evaluation of best model on test set
best_scale_prior_comb = tuning_results.sort_values("mape").iloc[0][
    ["yearly_seasonality_prior", "weekly_seasonality_prior"]
]
m_calls = Prophet(
    interval_width=0.95,
    yearly_seasonality=best_scale_prior_comb.iloc[0],
    weekly_seasonality=best_scale_prior_comb.iloc[1],
).fit(df_calls)
train_len = len(df_calls_train)
df_cv_calls = cross_validation(
    m_calls, initial=f"{train_len} days", period="30.5 days", horizon="45 days", disable_tqdm=True
)
df_p_calls = performance_metrics(df_cv_calls, rolling_window=1 / 45)  # rolling_window is in percent
df_p_calls = df_p_calls.groupby(df_p_calls["horizon"].dt.days).mean(numeric_only=True).reset_index()

In [ ]:
from prophet.plot import plot_cross_validation_metric

# show results of best model on test set
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
plot_cross_validation_metric(df_cv_calls, metric="mape", ax=axs[0], rolling_window=1 / 45)
axs[0].set_title("MAPE by horizon")
axs[0].set_ylabel("MAPE")
axs[1].plot(df_p_calls["horizon"], df_p_calls["coverage"])
axs[1].set_title("Coverage by horizon")
axs[1].set_xlabel("Horizon (days)")
axs[1].set_ylabel("Coverage")
axs[1].grid(True)
axs[2].hist(df_cv_calls["yhat"] - df_cv_calls["y"])
axs[2].grid()
axs[2].set_title("Residuals")
axs[2].set_xlabel("Error")
axs[2].set_ylabel("Frequency")
fig.tight_layout()

In [ ]:
# investigation of individual test sets (cutoffs) to show monthly seasonality
plt.close("all")
cutoffs = df_cv_calls["cutoff"].unique()
fig, axs = plt.subplots(4, len(cutoffs) // 3, figsize=(12, 6), sharex=True)
for i in range(len(cutoffs)):
    df_plot = df_cv_calls.where(df_cv_calls["cutoff"] == cutoffs[i]).dropna()
    x = range(len(df_plot))
    row = i // (len(cutoffs) // 3)
    col = i % (len(cutoffs) // 3)
    axs[row, col].plot(df_plot.loc[:, "yhat"].values, label=f"model")
    axs[row, col].plot(df_plot.loc[:, "y"].values, label=f"original")
    axs[row, col].grid()
    axs[row, col].legend(loc="upper left")
    axs[row, col].set_title(f"Cutoff {i+1}")
for col in range(len(cutoffs) // 3):
    axs[3, col].set_xlabel("Horizon (days)")
fig.tight_layout()

### Addition of monthly seasonality

Since there is a monthly pattern visible in the residuals and in the data that is not covered by the model, a monthly seasonality has to be added.

In [ ]:
# plot of all months shows that the data fits to a seasonality rather than a regressor
# even tho there is peak around 20th day of forecast (which corresponds to beginning of month (see cutoffs))
plt.close("all")
df_calls_plot = df_calls.copy()
df_calls_plot["ds"] = pd.to_datetime(df_calls_plot["ds"])
df_calls_plot["month"] = df_calls_plot["ds"].dt.to_period("M")
df_calls_plot["day"] = df_calls_plot["ds"].dt.day

fig, ax = plt.subplots(figsize=(8, 3))
for month, g in df_calls_plot.groupby("month"):
    ax.plot(g["day"], g["y"], linewidth=1, alpha=0.6, label=str(month))

ax.set_title("Calls by day-of-month (one line per month)")
ax.set_xlabel("Day of month")
ax.set_ylabel("Calls")
ax.grid()
fig.tight_layout()

In [ ]:
# cross validation for optimization of hyperparameters (weekly, monthly and yearly seasonality prior)
#   horizon: duration of use-case (always starting from the 15th, depending on starting month: 16d+30d or 15d+31d)
#   initial: 1 year to ensure that model can learn every seasonality
#   period: repeat for every month
import itertools
from prophet.diagnostics import cross_validation, performance_metrics

# generate scale_prior for weekly, monthly and yearly seasonality
scale_priors = [0.01, 0.1, 1, 10]
scale_prior_combs = list(itertools.product(scale_priors, repeat=3))

# use cross validation to evaluate all combinations of scale_prior
tuning_results = pd.DataFrame(
    columns=["weekly_seasonality_prior", "monthly_seasonality_prior", "yearly_seasonality_prior", "mape"]
)
for scale_prior_comb in scale_prior_combs:
    m = Prophet(interval_width=0.95, yearly_seasonality=scale_prior_comb[0], weekly_seasonality=scale_prior_comb[1])
    m.add_seasonality(name="monthly", period=30.5, fourier_order=10, prior_scale=scale_prior_comb[2])
    m.fit(df_calls_train)
    df_cv = cross_validation(m, initial="365.25 days", period="30.5 days", horizon="45 days", disable_tqdm=True)
    df_p = performance_metrics(df_cv)

    # store results
    tuning_results.loc[len(tuning_results)] = {
        "yearly_seasonality_prior": scale_prior_comb[0],
        "weekly_seasonality_prior": scale_prior_comb[1],
        "monthly_seasonality_prior": scale_prior_comb[2],
        "mape": np.mean(df_p["mape"].values),
    }

# show best results
display(tuning_results.sort_values("mape").head())

In [ ]:
# evaluation of best model on test set
best_scale_prior_comb = tuning_results.sort_values("mape").iloc[0][
    ["yearly_seasonality_prior", "weekly_seasonality_prior", "monthly_seasonality_prior"]
]
m_calls = Prophet(
    interval_width=0.95,
    yearly_seasonality=best_scale_prior_comb.iloc[0],
    weekly_seasonality=best_scale_prior_comb.iloc[1],
)
m_calls.add_seasonality(name="monthly", period=30.5, fourier_order=10, prior_scale=best_scale_prior_comb.iloc[2])
m_calls.fit(df_calls)
train_len = len(df_calls_train)
df_cv_calls = cross_validation(
    m_calls, initial=f"{train_len} days", period="30.5 days", horizon="45 days", disable_tqdm=True
)
df_p_calls = performance_metrics(df_cv_calls, rolling_window=1 / 45)  # rolling_window is in percent
df_p_calls = df_p_calls.groupby(df_p_calls["horizon"].dt.days).mean(numeric_only=True).reset_index()

In [ ]:
from prophet.plot import plot_cross_validation_metric

# show results of best model on test set
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
plot_cross_validation_metric(df_cv_calls, metric="mape", ax=axs[0], rolling_window=1 / 45)
axs[0].set_title("MAPE by horizon")
axs[0].set_ylabel("MAPE")
axs[1].plot(df_p_calls["horizon"], df_p_calls["coverage"])
axs[1].set_title("Coverage by horizon")
axs[1].set_xlabel("Horizon (days)")
axs[1].set_ylabel("Coverage")
axs[1].grid(True)
axs[2].hist(df_cv_calls["yhat"] - df_cv_calls["y"])
axs[2].grid()
axs[2].set_title("Residuals")
axs[2].set_xlabel("Error")
axs[2].set_ylabel("Frequency")
fig.tight_layout()

In [ ]:
# investigation of individual test sets (cutoffs) to show monthly seasonality disappeared
plt.close("all")
cutoffs = df_cv_calls["cutoff"].unique()
fig, axs = plt.subplots(4, len(cutoffs) // 3, figsize=(12, 6), sharex=True)
for i in range(len(cutoffs)):
    df_plot = df_cv_calls.where(df_cv_calls["cutoff"] == cutoffs[i]).dropna()
    x = range(len(df_plot))
    row = i // (len(cutoffs) // 3)
    col = i % (len(cutoffs) // 3)
    axs[row, col].plot(df_plot.loc[:, "yhat"].values, label=f"model")
    axs[row, col].plot(df_plot.loc[:, "y"].values, label=f"original")
    axs[row, col].grid()
    axs[row, col].legend(loc="upper left")
    axs[row, col].set_title(f"Cutoff {i+1}")
for col in range(len(cutoffs) // 3):
    axs[3, col].set_xlabel("Horizon (days)")
fig.tight_layout()

### Useful for final model

In [ ]:
# for predicting into the future

futures_dates = m.make_future_dataframe(periods=45, include_history=False)
forecast = m.predict(futures_dates)
m.plot(forecast)
plt.show()

## Train model to predict perc_sick

In [ ]:
# by default there are too many change points which do not represent data
#   therefore, change point prior scale is set to 0.01
from prophet.plot import add_changepoints_to_plot

fig, axs = plt.subplots(2, 1, figsize=(8, 6))

m = Prophet(interval_width=0.95)
m.fit(df_perc_sick)
forecast = m.predict(df_perc_sick)
m.plot(forecast, ax=axs[0])
add_changepoints_to_plot(axs[0], m, forecast)
axs[0].set_title("Changepoints of Prophet model for perc_sick w/o limitation")
axs[0].plot([], [], "r--", label="Changepoints")
axs[0].plot([], [], "r-", label="Trend")
axs[0].legend()

m = Prophet(interval_width=0.95, changepoint_prior_scale=0.01)
m.fit(df_perc_sick)
forecast = m.predict(df_perc_sick)
m.plot(forecast, ax=axs[1])
add_changepoints_to_plot(axs[1], m, forecast)
axs[1].set_title("Changepoints of Prophet model for perc_sick w/ limitation")
axs[1].plot([], [], "r--", label="Changepoints")
axs[1].plot([], [], "r-", label="Trend")
axs[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
# first model w/o hyperparameter optimization
m = Prophet(interval_width=0.95, changepoint_prior_scale=0.01)
m.fit(df_perc_sick_train)

forecast = m.predict(df_perc_sick_train)
print("MAE: ", np.mean(np.abs(forecast["yhat"].values - df_perc_sick_train["y"].values)))
fig = m.plot_components(forecast)
fig.suptitle("Components of Prophet model for perc_sick")
fig.tight_layout()
fig, ax = plt.subplots(figsize=(6, 3))
m.plot(forecast, ax=ax)
ax.scatter(df_perc_sick_train["ds"], df_perc_sick_train["y"], color="red")
ax.set_title("Forecast vs Actuals")
ax.legend()
fig.tight_layout()

In [ ]:
# tryout of adding quarterly seasonality
#   determined in FFT, but there is no real improvement in MAE
m = Prophet(interval_width=0.95, changepoint_prior_scale=0.01)
m.add_seasonality(name="quarterly", period=91.25, fourier_order=10)
m.fit(df_perc_sick_train)

forecast = m.predict(df_perc_sick_train)
print("MAE: ", np.mean(np.abs(forecast["yhat"].values - df_perc_sick_train["y"].values)))
fig = m.plot_components(forecast)
fig.suptitle("Components of Prophet model for perc_sick")
fig.tight_layout()

In [ ]:
# cross validation for optimization of hyperparameters (weekly and yearly seasonality prior)
#   horizon: duration of use-case (always starting from the 15th, depending on starting month: 16d+30d or 15d+31d)
#   initial: 1 year to ensure that model can learn every seasonality
#   period: repeat for every month
import itertools
from prophet.diagnostics import cross_validation, performance_metrics

# generate scale_prior for weekly and yearly seasonality
scale_priors = [0.01, 0.1, 1, 10]
scale_prior_combs = list(itertools.product(scale_priors, repeat=2))

# use cross validation to evaluate all combinations of scale_prior
tuning_results = pd.DataFrame(columns=["weekly_seasonality_prior", "yearly_seasonality_prior", "mae"])
for scale_prior_comb in scale_prior_combs:
    m = Prophet(
        interval_width=0.95, yearly_seasonality=scale_prior_comb[0], weekly_seasonality=scale_prior_comb[1]
    ).fit(df_perc_sick_train)
    df_cv = cross_validation(m, initial="365.25 days", period="30.5 days", horizon="45 days", disable_tqdm=True)
    df_p = performance_metrics(df_cv)

    # store results
    tuning_results.loc[len(tuning_results)] = {
        "yearly_seasonality_prior": scale_prior_comb[0],
        "weekly_seasonality_prior": scale_prior_comb[1],
        "mae": np.mean(df_p["mae"].values),
    }

# show best results
display(tuning_results.sort_values("mae").head())

In [ ]:
# evaluation of best model on test set
best_scale_prior_comb = tuning_results.sort_values("mae").iloc[0][
    ["yearly_seasonality_prior", "weekly_seasonality_prior"]
]
m_perc_sick = Prophet(
    interval_width=0.95,
    yearly_seasonality=best_scale_prior_comb.iloc[0],
    weekly_seasonality=best_scale_prior_comb.iloc[1],
).fit(df_perc_sick)
train_len = len(df_perc_sick_train)
df_cv_perc_sick = cross_validation(
    m_perc_sick, initial=f"{train_len} days", period="30.5 days", horizon="45 days", disable_tqdm=True
)
df_p_perc_sick = performance_metrics(df_cv_perc_sick, rolling_window=1 / 45)  # rolling_window is in percent
df_p_perc_sick = df_p_perc_sick.groupby(df_p_perc_sick["horizon"].dt.days).mean(numeric_only=True).reset_index()

In [ ]:
from prophet.plot import plot_cross_validation_metric

# show results of best model on test set
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
plot_cross_validation_metric(df_cv_perc_sick, metric="mae", ax=axs[0])
axs[0].set_title("MAE by horizon")
axs[0].set_ylabel("MAE")
axs[1].plot(df_p_perc_sick["coverage"])
axs[1].set_title("Coverage by horizon")
axs[1].set_xlabel("Horizon (days)")
axs[1].set_ylabel("Coverage")
axs[1].grid(True)
axs[2].hist(df_cv_perc_sick["yhat"] - df_cv_perc_sick["y"])
axs[2].set_title("Residuals")
axs[2].set_xlabel("Error")
axs[2].set_ylabel("Frequency")
fig.tight_layout()

# Archiv

In [ ]:
# tryout of adding regressor for beginning of month
#   determined in FFT, but only results in 1% improvement in MAPE and therefore not worth the extra complexity
df_calls.loc[:, "start_of_month"] = (pd.to_datetime(df_calls["ds"]).dt.day <= 2).astype(int)
df_calls_test.loc[:, "start_of_month"] = (pd.to_datetime(df_calls_test["ds"]).dt.day <= 2).astype(int)
m = Prophet(interval_width=0.95)
m.add_regressor("start_of_month")
m.fit(df_calls)
forecast = m.predict(df_calls_test)
print("MAPE: ", np.mean(np.abs(forecast["yhat"].values - df_calls_test["y"].values) / df_calls_test["y"].values))
fig = m.plot_components(forecast)